# 04 — Hypothesis Testing

**Goal**: Use statistical tests to check whether the patterns we saw in EDA are real or just noise.

**What this notebook covers**:
1. Do verified leads really convert more? (Z-test)
2. Does keyword group affect RPL? (Kruskal-Wallis)
3. A/B test sample size calculation

All tests use **α = 0.05** (95% confidence level).

### What is a sample size calculation?

Before launching an A/B test, you need to know **how many users you need per group** to reliably detect the effect you care about. Run too few and the test is inconclusive; run too many and you waste traffic and time.

Three inputs decide the answer:
- **Baseline rate** — what's the current conversion rate?
- **Minimum detectable effect (MDE)** — the smallest improvement you'd care about (e.g. +1pp).
- **Power & significance** — typically 80% power and 5% significance.

The output is the sample size per arm. If it's bigger than your weekly traffic, the test isn't feasible — better to know that *before* you launch.

📺 *Visual explainer:* [StatQuest — Statistical Power](https://www.youtube.com/watch?v=Rsc5znwR5FA)

### What is a Kruskal-Wallis test?

We have **more than two groups** (seven keyword groups) and we want to know whether their RPL distributions differ. We can't use a Z-test (only handles two groups) and we can't use a standard ANOVA because RPL is heavily skewed (most leads have £0 premium, a few have thousands).

**Kruskal-Wallis** is the non-parametric cousin of one-way ANOVA: it works on **ranks** instead of raw values, so skew and outliers don't break it. It tests whether at least one group's distribution differs from the rest — but it does **not** tell you *which* one (you'd need post-hoc tests for that).

📺 *Visual explainer:* [StatQuest — Kruskal-Wallis test](https://www.youtube.com/watch?v=nrK-jU5nqWE)

### What is a Z-test for two proportions?

When we compare two **rates** (conversion rate of verified vs unverified leads), we can't just say "23% > 18%, done." The gap could be real or could be noise from sample size. A **two-proportion Z-test** asks: *if both groups truly converted at the same rate, how unlikely is a gap this big by chance?*

- The **z-statistic** measures how many standard errors the observed gap is away from zero.
- The **p-value** is the probability of seeing a gap that big (or bigger) if there were really no difference.
- If `p < 0.05`, we reject the null hypothesis — the gap is unlikely to be noise.

📺 *Visual explainer:* [StatQuest — p-values clearly explained](https://www.youtube.com/watch?v=vemZtEM63GY)

In [1]:
import numpy as np
import pandas as pd
import pathlib
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.power import NormalIndPower

DATA = pathlib.Path("../data")
ALPHA = 0.05

leads = pd.read_parquet(DATA / "staging/leads_clean.parquet")
print(f"Leads: {leads.shape[0]:,} rows")

Leads: 7,485 rows


## 1 · Insurance Leads — Do verified leads really convert more?

In EDA we saw verified leads had a much higher conversion rate. Let's test it.

**Z-test for two proportions**:
- H₀: Conversion rate is the same for verified and unverified leads
- H₁: They differ

In [2]:
verified = leads[leads["verification_status"] == "verified"]["converted"]
unverified = leads[leads["verification_status"] == "not_verified"]["converted"]

n = np.array([len(verified), len(unverified)])
x = np.array([verified.sum(), unverified.sum()])
rates = x / n

z_stat, p_value = proportions_ztest(x, n)

print("Z-TEST: Verified vs Unverified Conversion")
print("=" * 45)
print(f"  Verified:   {rates[0]:.1%} ({n[0]:,} leads)")
print(f"  Unverified: {rates[1]:.1%} ({n[1]:,} leads)")
print(f"  Difference: {rates[0] - rates[1]:+.1%}")
print(f"  z-stat:     {z_stat:.3f}")
print(f"  p-value:    {p_value:.6f}")
print(f"  Significant: {'Yes' if p_value < ALPHA else 'No'} (α = {ALPHA})")

Z-TEST: Verified vs Unverified Conversion
  Verified:   9.5% (1,554 leads)
  Unverified: 3.7% (5,931 leads)
  Difference: +5.8%
  z-stat:     9.344
  p-value:    0.000000
  Significant: Yes (α = 0.05)


## 2 · Does keyword group affect RPL?

We saw different keyword groups had very different RPLs in EDA. Is this real?

**Kruskal-Wallis test** (non-parametric, works for multiple groups):
- H₀: All keyword groups have the same RPL distribution
- H₁: At least one group differs

In [3]:
# RPL at individual lead level = their premium (revenue per that one lead)
groups = [g["premium"].values for _, g in leads.groupby("keyword_group")]
h_stat, p_value = stats.kruskal(*groups)

print("KRUSKAL-WALLIS: Keyword Group RPL")
print("=" * 45)
print(f"  H-statistic: {h_stat:.2f}")
print(f"  p-value:     {p_value:.6f}")
print(f"  Significant: {'Yes' if p_value < ALPHA else 'No'}")

# Show group averages for context
kw = leads.groupby("keyword_group")["premium"].agg(["count", "median", "mean"]).round(2)
kw.columns = ["Leads", "Median RPL £", "Mean RPL £"]
print()
print(kw.sort_values("Mean RPL £", ascending=False).to_string())

KRUSKAL-WALLIS: Keyword Group RPL
  H-statistic: 19.71
  p-value:     0.003123
  Significant: Yes

                           Leads  Median RPL £  Mean RPL £
keyword_group                                             
Brand: Other                  89           0.0      252.59
Price / Quote Intent         419           0.0       85.87
Generic: Health Insurance   1219           0.0       75.34
Generic: Private Health     3484           0.0       75.11
Brand: Bupa                 1594           0.0       74.99
Comparison / Research        488           0.0       74.85
Other                        192           0.0       27.83


## 3 · A/B Test Example — Sample Size

If we wanted to run an A/B test to measure a 5 percentage point lift in conversion, how many leads do we need?

Uses the **Z-test power analysis** at 80% power, α = 0.05.

In [4]:
baseline_conv = leads["converted"].mean()
lift = 0.05  # detect a 5pp improvement

# Cohen's h (effect size for proportions)
p_treatment = baseline_conv + lift
effect_size = 2 * (np.arcsin(np.sqrt(p_treatment)) - np.arcsin(np.sqrt(baseline_conv)))

n_per_group = int(np.ceil(
    NormalIndPower().solve_power(effect_size=effect_size, alpha=0.05, power=0.80, alternative="two-sided")
))

print("A/B TEST SAMPLE SIZE")
print("=" * 45)
print(f"  Baseline conversion: {baseline_conv:.1%}")
print(f"  Minimum detectable lift: +{lift:.0%}")
print(f"  Target conversion: {baseline_conv + lift:.1%}")
print(f"  ─────────────────────────")
print(f"  Needed per group:  {n_per_group:,} leads")
print(f"  Total needed:      {n_per_group * 2:,} leads")
print(f"  At 50 leads/day:   ~{n_per_group * 2 / 50:.0f} days")

A/B TEST SAMPLE SIZE
  Baseline conversion: 4.9%
  Minimum detectable lift: +5%
  Target conversion: 9.9%
  ─────────────────────────
  Needed per group:  419 leads
  Total needed:      838 leads
  At 50 leads/day:   ~17 days


## Summary

| Finding | Test | Significant? |
|---------|------|:---:|
| Verified leads convert more | Z-test (proportions) | ✅ |
| Keyword groups differ in RPL | Kruskal-Wallis | ✅ |

**A/B test insight**: To detect a meaningful lift, we need a decent sample size — running tests too early gives unreliable results.

**Next**: Notebook 05 → ML modeling (predict which leads will convert).